In [1]:
#CORRECT MODEL
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

df = pd.read_csv('dataset_thesis_new_hdi_v8.csv')

#  2024 Team Sizes
team_2024_data = {
    'USA': 592, 'FRA': 573, 'AUS': 460, 'GBR': 327, 'CHN': 388, 'DEU': 428, 'JPN': 403,
    'ITA': 371, 'ESP': 383, 'CAN': 315, 'NLD': 258, 'BRA': 277, 'KOR': 141, 'NZL': 195,
    'HUN': 170, 'ZAF': 138, 'MEX': 107, 'UKR': 140, 'CUB': 61, 'POL': 210, 'CHE': 127,
    'SWE': 117, 'BEL': 165, 'IRL': 133, 'ARG': 136, 'UZB': 86, 'IRN': 40, 'KAZ': 79,
    'KEN': 72, 'JAM': 58, 'ETH': 34, 'NOR': 107, 'DNK': 124, 
    'RUS': 335, 'BLR': 101  
}
for iso, size in team_2024_data.items():
    df.loc[(df['iso_code_mapped'] == iso) & (df['year'] == 2024), 'Team_Size'] = size
    
df = df.sort_values(['iso_code_mapped', 'year'])

df['Team_Size'] = df.groupby('iso_code_mapped')['Team_Size'].ffill()

# Russia/Belarus 2024 Medal Override
df.loc[(df['iso_code_mapped'] == 'RUS') & (df['year'] == 2024), 'Total'] = 71
df.loc[(df['iso_code_mapped'] == 'BLR') & (df['year'] == 2024), 'Total'] = 7

# Create Empty 2028 Rows
countries = df['iso_code_mapped'].unique()
df_2028 = pd.DataFrame({'iso_code_mapped': countries, 'year': 2028})
df = pd.concat([df, df_2028], ignore_index=True)
df = df.sort_values(['iso_code_mapped', 'year']).reset_index(drop=True)

#  LOCF for Structural Variables (Fills 2024 gaps and 2028 rows)
for col in ['hdi', 'gdi', 'population', 'GDP_merged', 'perc_athletic_prime']:
    df[col] = df.groupby('iso_code_mapped')[col].ffill()

#  Lags (t-1 and t-4)
for col in ['GDP_merged', 'population', 'hdi', 'gdi', 'perc_athletic_prime']:
    df[f'{col}_t_1'] = df.groupby('iso_code_mapped')[col].shift(1)

df['GDP_t_4'] = df.groupby('iso_code_mapped')['GDP_merged'].shift(4)
df['GDP_Growth_Cycle'] = ((df['GDP_merged_t_1'] - df['GDP_t_4']) / df['GDP_t_4']) * 100

olympic_years = [1996, 2000, 2004, 2008, 2012, 2016, 2021, 2024, 2028]
oly = df[df['year'].isin(olympic_years)].copy()
oly = oly.sort_values(['iso_code_mapped', 'year'])

oly['Lagged_Team_Size'] = oly.groupby('iso_code_mapped')['Team_Size'].shift(1)
oly['Prev_Total_Medals'] = oly.groupby('iso_code_mapped')['Total'].shift(1)

#  Prep Features
oly['Log_GDP_t_1'] = np.log10(oly['GDP_merged_t_1'])
oly['Log_Pop_t_1'] = np.log10(oly['population_t_1'])
oly['Is_Host'] = np.where((oly['iso_code_mapped'] == 'USA') & (oly['year'] == 2028), 1, oly['Is_Host'])

final_features = [
    'Log_GDP_t_1', 'Log_Pop_t_1', 'GDP_Growth_Cycle', 
    'perc_athletic_prime_t_1', 'Lagged_Team_Size', 'Prev_Total_Medals', 
    'hdi_t_1', 'gdi_t_1', 'Is_Host'
]

for f in final_features:
    if f in ['Lagged_Team_Size', 'Prev_Total_Medals', 'Is_Host']:
        oly[f] = oly[f].fillna(0)
    else:
        oly[f] = oly[f].fillna(oly[f].mean())

#  Train directly on 'Total' (Raw Medals)
train = oly[(oly['year'] < 2028) & oly['Total'].notna()]
predict_2028 = oly[oly['year'] == 2028].copy()

X_train = train[final_features]
y_train = train['Total'] # DIRECT RAW MEDAL TARGET
X_predict = predict_2028[final_features]

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

#  Predict and Output
predictions = rf.predict(X_predict)
predict_2028['Predicted_Total_Medals'] = predictions

# Calculate Medal Share (%) based on predicted totals matching the ~1044 medals available
total_predicted = predictions.sum()
predict_2028['Predicted_Medal_Share_Pct'] = (predictions / total_predicted) * 100

output = predict_2028[['iso_code_mapped', 'Predicted_Total_Medals', 'Predicted_Medal_Share_Pct']].sort_values(by='Predicted_Total_Medals', ascending=False).head(30)
output['Predicted_Total_Medals'] = output['Predicted_Total_Medals'].round(1)
output['Predicted_Medal_Share_Pct'] = output['Predicted_Medal_Share_Pct'].round(2)

print("\n--- LA 2028 PREDICTIONS (COMPLETE MODEL) ---")
print(output.to_string(index=False))

# Calculate Permutation Importance for this exact model (using the training data since 2028 is unknown)
perm = permutation_importance(rf, X_train, y_train, n_repeats=10, random_state=42)
imp_df = pd.DataFrame({
    'Feature': final_features,
    'Importance_Mean': perm.importances_mean
}).sort_values('Importance_Mean', ascending=False)
print("\n--- FEATURE IMPORTANCE (COMPLETE MODEL TRAINED UP TO 2024) ---")
print(imp_df.to_string(index=False))


--- LA 2028 PREDICTIONS (COMPLETE MODEL) ---
iso_code_mapped  Predicted_Total_Medals  Predicted_Medal_Share_Pct
            USA                   118.0                      10.12
            CHN                    93.7                       8.04
            RUS                    70.6                       6.06
            FRA                    63.9                       5.48
            GBR                    63.6                       5.45
            AUS                    51.3                       4.40
            JPN                    43.4                       3.72
            DEU                    39.6                       3.40
            ITA                    38.9                       3.34
            NLD                    35.1                       3.01
            KOR                    31.4                       2.69
            CAN                    29.4                       2.53
            BRA                    24.0                       2.06
            ESP 

In [6]:
#MODEL WITH RUSSIA STILL BANNED
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

#  PREP THE  DATA 
df = pd.read_csv('dataset_thesis_new_hdi_v8.csv')
df = df.sort_values(['iso_code_mapped', 'year'])

# Use Inclusion Overrides for Training
df.loc[(df['iso_code_mapped'] == 'RUS') & (df['year'] == 2021), 'Team_Size'] = 335
df.loc[(df['iso_code_mapped'] == 'RUS') & (df['year'] == 2024), 'Total'] = 71

# Bridge the gaps (ffill)
cols_to_fix = ['GDP_merged', 'population', 'hdi', 'gdi', 'perc_athletic_prime', 'Team_Size', 'Total']
for col in cols_to_fix:
    df[col] = df.groupby('iso_code_mapped')[col].ffill()

# CREATE LAGS & FEATURES
for col in ['GDP_merged', 'population', 'hdi', 'gdi', 'perc_athletic_prime']:
    df[f'{col}_t_1'] = df.groupby('iso_code_mapped')[col].shift(1)

df['GDP_t_4'] = df.groupby('iso_code_mapped')['GDP_merged'].shift(4)
df['GDP_Growth_Cycle'] = ((df['GDP_merged_t_1'] - df['GDP_t_4']) / df['GDP_t_4']) * 100

oly = df[df['year'].isin([1996, 2000, 2004, 2008, 2012, 2016, 2021, 2024])].copy()
oly['Lagged_Team_Size'] = oly.groupby('iso_code_mapped')['Team_Size'].shift(1)
oly['Prev_Total_Medals'] = oly.groupby('iso_code_mapped')['Total'].shift(1)

# Log features
oly['Log_GDP_t_1'] = np.log10(oly['GDP_merged_t_1'])
oly['Log_Pop_t_1'] = np.log10(oly['population_t_1'])

final_features = ['Log_GDP_t_1', 'Log_Pop_t_1', 'GDP_Growth_Cycle', 'perc_athletic_prime_t_1', 
                  'Lagged_Team_Size', 'Prev_Total_Medals', 'hdi_t_1', 'gdi_t_1', 'Is_Host']

# Impute
for f in final_features:
    oly[f] = oly[f].fillna(0) if f in ['Lagged_Team_Size', 'Prev_Total_Medals', 'Is_Host'] else oly[f].fillna(oly[f].mean())

# TRAIN THE "INCLUSION BRAIN"
rf_brain = RandomForestRegressor(n_estimators=100, random_state=42)
rf_brain.fit(oly[final_features], oly['Total'])

#  PREDICT SCENARIO B (EXCLUSION WORLD)
# Create the 2028 row
predict_2028_excl = oly[oly['year'] == 2024].copy() # Using 2024 data as the basis for 2028 lags
predict_2028_excl['year'] = 2028

# FORCE THE EXCLUSION: Set Russia and Belarus to zero capacity
predict_2028_excl.loc[predict_2028_excl['iso_code_mapped'].isin(['RUS', 'BLR']), ['Prev_Total_Medals', 'Lagged_Team_Size']] = 0
# SET THE HOST: USA 2028
predict_2028_excl['Is_Host'] = np.where(predict_2028_excl['iso_code_mapped'] == 'USA', 1, 0)

# Run the prediction
raw_preds = rf_brain.predict(predict_2028_excl[final_features])
predict_2028_excl['Predicted_Raw'] = raw_preds

#  REDISTRIBUTION (Zero-Sum Normalization)
olympic_total = 1044
ratio = olympic_total / raw_preds.sum()
predict_2028_excl['Redistributed'] = raw_preds * ratio
predict_2028_excl['Share'] = (predict_2028_excl['Redistributed'] / olympic_total) * 100

# Final Results
output = predict_2028_excl[['iso_code_mapped', 'Redistributed', 'Share']].sort_values('Redistributed', ascending=False).head(15)
print(f"\n--- LA 2028 EXCLUSION (Correct Scenario Logic) ---")
print(f"Redistribution Ratio: {ratio:.4f}")
print(output.round(2).to_string(index=False))


--- LA 2028 EXCLUSION (Correct Scenario Logic) ---
Redistribution Ratio: 1.0085
iso_code_mapped  Redistributed  Share
            USA         122.51  11.73
            CHN          92.30   8.84
            GBR          65.35   6.26
            JPN          49.89   4.78
            AUS          48.01   4.60
            FRA          44.63   4.27
            ITA          42.02   4.02
            DEU          36.20   3.47
            NLD          34.65   3.32
            KOR          29.11   2.79
            CAN          27.26   2.61
            BRA          20.59   1.97
            NZL          19.99   1.91
            ESP          19.43   1.86
            HUN          18.77   1.80


In [2]:
#CORRECT MODEL
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

df = pd.read_csv('dataset_thesis_new_hdi_v8.csv')

#  2024 Team Sizes
team_2024_data = {
    'USA': 592, 'FRA': 573, 'AUS': 460, 'GBR': 327, 'CHN': 388, 'DEU': 428, 'JPN': 403,
    'ITA': 371, 'ESP': 383, 'CAN': 315, 'NLD': 258, 'BRA': 277, 'KOR': 141, 'NZL': 195,
    'HUN': 170, 'ZAF': 138, 'MEX': 107, 'UKR': 140, 'CUB': 61, 'POL': 210, 'CHE': 127,
    'SWE': 117, 'BEL': 165, 'IRL': 133, 'ARG': 136, 'UZB': 86, 'IRN': 40, 'KAZ': 79,
    'KEN': 72, 'JAM': 58, 'ETH': 34, 'NOR': 107, 'DNK': 124, 
    'RUS': 335, 'BLR': 101  
}
for iso, size in team_2024_data.items():
    df.loc[(df['iso_code_mapped'] == iso) & (df['year'] == 2024), 'Team_Size'] = size
    
df = df.sort_values(['iso_code_mapped', 'year'])

df['Team_Size'] = df.groupby('iso_code_mapped')['Team_Size'].ffill()

# Russia/Belarus 2024 Medal Override
df.loc[(df['iso_code_mapped'] == 'RUS') & (df['year'] == 2024), 'Total'] = 71
df.loc[(df['iso_code_mapped'] == 'BLR') & (df['year'] == 2024), 'Total'] = 7

# Create Empty 2028 Rows
countries = df['iso_code_mapped'].unique()
df_2028 = pd.DataFrame({'iso_code_mapped': countries, 'year': 2028})
df = pd.concat([df, df_2028], ignore_index=True)
df = df.sort_values(['iso_code_mapped', 'year']).reset_index(drop=True)

#  LOCF for Structural Variables (Fills 2024 gaps and 2028 rows)
for col in ['hdi', 'gdi', 'population', 'GDP_merged', 'perc_athletic_prime']:
    df[col] = df.groupby('iso_code_mapped')[col].ffill()

#  Lags (t-1 and t-4)
for col in ['GDP_merged', 'population', 'hdi', 'gdi', 'perc_athletic_prime']:
    df[f'{col}_t_1'] = df.groupby('iso_code_mapped')[col].shift(1)

df['GDP_t_4'] = df.groupby('iso_code_mapped')['GDP_merged'].shift(4)
df['GDP_Growth_Cycle'] = ((df['GDP_merged_t_1'] - df['GDP_t_4']) / df['GDP_t_4']) * 100

olympic_years = [1996, 2000, 2004, 2008, 2012, 2016, 2021, 2024, 2028]
oly = df[df['year'].isin(olympic_years)].copy()
oly = oly.sort_values(['iso_code_mapped', 'year'])

oly['Lagged_Team_Size'] = oly.groupby('iso_code_mapped')['Team_Size'].shift(1)
oly['Prev_Total_Medals'] = oly.groupby('iso_code_mapped')['Total'].shift(1)

#  Prep Features
oly['Log_GDP_t_1'] = np.log10(oly['GDP_merged_t_1'])
oly['Log_Pop_t_1'] = np.log10(oly['population_t_1'])
#oly['Is_Host'] = np.where((oly['iso_code_mapped'] == 'USA') & (oly['year'] == 2028), 1, oly['Is_Host'])

final_features = [
    'Log_GDP_t_1', 'Log_Pop_t_1', 'GDP_Growth_Cycle', 
    'perc_athletic_prime_t_1', 'Lagged_Team_Size', 'Prev_Total_Medals', 
    'hdi_t_1', 'gdi_t_1'
]

for f in final_features:
    if f in ['Lagged_Team_Size', 'Prev_Total_Medals']:
        oly[f] = oly[f].fillna(0)
    else:
        oly[f] = oly[f].fillna(oly[f].mean())

#  Train directly on 'Total' (Raw Medals)
train = oly[(oly['year'] < 2028) & oly['Total'].notna()]
predict_2028 = oly[oly['year'] == 2028].copy()

X_train = train[final_features]
y_train = train['Total'] # DIRECT RAW MEDAL TARGET
X_predict = predict_2028[final_features]

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

#  Predict and Output
predictions = rf.predict(X_predict)
predict_2028['Predicted_Total_Medals'] = predictions

# Calculate Medal Share (%) based on predicted totals matching the ~1044 medals available
total_predicted = predictions.sum()
predict_2028['Predicted_Medal_Share_Pct'] = (predictions / total_predicted) * 100

output = predict_2028[['iso_code_mapped', 'Predicted_Total_Medals', 'Predicted_Medal_Share_Pct']].sort_values(by='Predicted_Total_Medals', ascending=False).head(30)
output['Predicted_Total_Medals'] = output['Predicted_Total_Medals'].round(1)
output['Predicted_Medal_Share_Pct'] = output['Predicted_Medal_Share_Pct'].round(2)

print("\n--- LA 2028 PREDICTIONS (COMPLETE MODEL) ---")
print(output.to_string(index=False))

# Calculate Permutation Importance for this exact model (using the training data since 2028 is unknown)
perm = permutation_importance(rf, X_train, y_train, n_repeats=10, random_state=42)
imp_df = pd.DataFrame({
    'Feature': final_features,
    'Importance_Mean': perm.importances_mean
}).sort_values('Importance_Mean', ascending=False)
print("\n--- FEATURE IMPORTANCE (COMPLETE MODEL TRAINED UP TO 2024) ---")
print(imp_df.to_string(index=False))


--- LA 2028 PREDICTIONS (COMPLETE MODEL) ---
iso_code_mapped  Predicted_Total_Medals  Predicted_Medal_Share_Pct
            USA                   118.2                      10.05
            CHN                    95.1                       8.08
            RUS                    71.5                       6.07
            FRA                    64.2                       5.46
            GBR                    63.6                       5.41
            AUS                    50.5                       4.29
            JPN                    48.2                       4.10
            ITA                    40.7                       3.46
            DEU                    40.6                       3.45
            NLD                    35.6                       3.02
            KOR                    30.8                       2.62
            CAN                    30.0                       2.55
            BRA                    24.5                       2.08
            NZL 